In [ ]:
import os
import re
import joblib
import pandas as pd
import numpy as np
import torch  # For GPU check (embeddings)

# Embeddings
from sentence_transformers import SentenceTransformer

# 10 Traditional ML Algorithms
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, HistGradientBoostingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import (
    classification_report, precision_score,
    recall_score, f1_score
)

# For GPU in TensorFlow, though we only do embeddings/ML here
import tensorflow as tf

# =========================================================
# 0. GPU CHECK
# =========================================================
if tf.config.list_physical_devices('GPU'):
    print("TensorFlow sees a GPU available (for TF operations).")
else:
    print("TensorFlow is using CPU (no GPU found).")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"SentenceTransformer will use device: {device}")

# =========================================================
# 1. LOAD & COMBINE DATA
# =========================================================
TRAIN_CSV = "/content/drive/MyDrive/train.csv"
VAL_CSV   = "/content/drive/MyDrive/val.csv"
TEST_CSV  = "/content/drive/MyDrive/test.csv"

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

# We assume columns "SDG 1..17" are 0/1 floats -> convert to int if needed
sdg_cols = [f"SDG {i}" for i in range(1, 18)]
train_df[sdg_cols] = train_df[sdg_cols].astype(int)
val_df[sdg_cols]   = val_df[sdg_cols].astype(int)
test_df[sdg_cols]  = test_df[sdg_cols].astype(int)

df_train = pd.concat([train_df, val_df]).reset_index(drop=True)
df_test  = test_df.copy()

print("Train size:", df_train.shape, "Test size:", df_test.shape)

# =========================================================
# 2. TEXT PREPROCESSING
# =========================================================
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text

df_train["clean_text"] = df_train["text"].apply(clean_text)
df_test["clean_text"]  = df_test["text"].apply(clean_text)

X_train_text = df_train["clean_text"].values
y_train      = df_train[sdg_cols].values
X_test_text  = df_test["clean_text"].values
y_test       = df_test[sdg_cols].values

# =========================================================
# 3. EMBEDDINGS: LOAD OR COMPUTE
# =========================================================
EMBED_TRAIN_PATH = "/content/drive/MyDrive/X_train_emb.npy"
EMBED_TEST_PATH  = "/content/drive/MyDrive/X_test_emb.npy"

if os.path.exists(EMBED_TRAIN_PATH) and os.path.exists(EMBED_TEST_PATH):
    print("Found saved embeddings in Drive. Loading...")
    X_train_emb = np.load(EMBED_TRAIN_PATH)
    X_test_emb  = np.load(EMBED_TEST_PATH)
    print("Train Emb shape:", X_train_emb.shape, "Test Emb shape:", X_test_emb.shape)
else:
    print("No saved embeddings found. Computing with SentenceTransformer...")

    embedder = SentenceTransformer(
        'sentence-transformers/all-MiniLM-L6-v2',
        device=device  # 'cuda' if GPU, else 'cpu'
    )
    X_train_emb = embedder.encode(X_train_text, show_progress_bar=True)
    X_test_emb  = embedder.encode(X_test_text,  show_progress_bar=True)

    np.save(EMBED_TRAIN_PATH, X_train_emb)
    np.save(EMBED_TEST_PATH, X_test_emb)
    print("Saved embeddings to Drive for next time.")

# =========================================================
# 4. MULTI-LABEL METRIC HELPERS
# =========================================================
def compute_multi_label_metrics(y_true, y_pred):
    """
    Return a dict with micro/macro precision, recall, f1 in a multi-label setting.
    """
    results = {}
    y_true_1d = y_true.ravel()
    y_pred_1d = y_pred.ravel()

    results["micro_precision"] = precision_score(y_true_1d, y_pred_1d, average="micro")
    results["micro_recall"]    = recall_score(y_true_1d, y_pred_1d, average="micro")
    results["micro_f1"]        = f1_score(y_true_1d, y_pred_1d, average="micro")

    results["macro_precision"] = precision_score(y_true_1d, y_pred_1d, average="macro")
    results["macro_recall"]    = recall_score(y_true_1d, y_pred_1d, average="macro")
    results["macro_f1"]        = f1_score(y_true_1d, y_pred_1d, average="macro")
    return results

def print_metrics_dict(model_name, metrics):
    print(f"=== {model_name} Metrics ===")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")
    print("-"*50)

# =========================================================
# 5. DEFINE 10 ML ALGORITHMS & TRAIN
# =========================================================
ml_algos = {
    "BernoulliNB": BernoulliNB(),
    "RandomForest": RandomForestClassifier(n_estimators=100, n_jobs=-1),
    "KNeighbors": KNeighborsClassifier(n_neighbors=5),
    "SVC": SVC(kernel='rbf', probability=True),
    "MLP": MLPClassifier(hidden_layer_sizes=(128,), max_iter=300),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "DecisionTree": DecisionTreeClassifier(),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=100, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100),
    "HistGradientBoosting": HistGradientBoostingClassifier()
}

ml_models = []
ml_results = []

for name, base_clf in ml_algos.items():
    print(f"\nTraining {name} ...")
    multi_label_clf = MultiOutputClassifier(base_clf)

    try:
        multi_label_clf.fit(X_train_emb, y_train)
    except ValueError as e:
        # If any label has a single class => model can fail (like SVC).
        print(f"Skipping {name} due to error:\n  {e}")
        continue

    y_pred_test = multi_label_clf.predict(X_test_emb)
    metrics_dict = compute_multi_label_metrics(y_test, y_pred_test)
    ml_results.append((name, metrics_dict))
    ml_models.append((name, multi_label_clf))

    print_metrics_dict(name, metrics_dict)

# =========================================================
# 6. FIND BEST MODEL BY micro_f1
# =========================================================
def get_micro_f1(entry):
    # entry is (name, metrics_dict)
    return entry[1]["micro_f1"]

ml_results.sort(key=get_micro_f1, reverse=True)

print("\n==== Final Results ====")
for (model_name, met) in ml_results:
    print_metrics_dict(model_name, met)

best_model_name, best_metrics = ml_results[0]
print(f"\nBest Overall Model: {best_model_name} (micro-F1={best_metrics['micro_f1']:.4f})")

def get_best_model(name):
    for nm, model_obj in ml_models:
        if nm == name:
            return model_obj
    return None

best_model = get_best_model(best_model_name)

# =========================================================
# 7. CLASSIFICATION REPORT
# =========================================================
y_pred_best = best_model.predict(X_test_emb)
print(f"\nClassification Report ({best_model_name}):")
print(classification_report(y_test, y_pred_best, target_names=sdg_cols))

# =========================================================
# 8. SAVE BEST MODEL
# =========================================================
MODEL_SAVE_PATH = "/content/drive/MyDrive/best_overall_model.pkl"
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

print(f"\nSaving best ML model ({best_model_name}) to {MODEL_SAVE_PATH}")
joblib.dump(best_model, MODEL_SAVE_PATH)

# =========================================================
# 9. COVERAGE PROBABILITIES & LLM-STYLE DEMO
# =========================================================
def predict_coverage_prob(model_name, texts):
    """
    Returns predicted coverage probabilities in [0..1]
    for each of the 17 SDGs, using the best model's
    sub-estimators' predict_proba() for "1" class.

    If a sub-estimator was trained on only one class (all 0s or all 1s),
    predict_proba() will return only one column. In that case:
       - If classes_[0] == 1 -> probability for '1' is 1.0
       - If classes_[0] == 0 -> probability for '1' is 0.0
    """
    cleaned = [clean_text(t) for t in texts]
    obj = get_best_model(model_name)
    if obj is None:
        raise ValueError(f"No model found for {model_name}")

    # We'll create a fresh embedder for new text
    embedder_gpu = SentenceTransformer(
        'sentence-transformers/all-MiniLM-L6-v2',
        device=device
    )
    X_emb = embedder_gpu.encode(cleaned)

    coverage_matrix = []
    for estimator in obj.estimators_:
        classes_ = estimator.classes_
        # shape (n_samples, 1 or 2) => we want column 1 probability if possible
        if len(classes_) == 1:
            # Only one class known to this sub-estimator
            if classes_[0] == 1:
                # Always predict class "1" with probability 1
                label_prob = np.ones(X_emb.shape[0])
            else:
                # Always predict class "0" with probability 0
                label_prob = np.zeros(X_emb.shape[0])
        else:
            # Normal case: 2 classes [0, 1]
            label_prob = estimator.predict_proba(X_emb)[:, 1]
        coverage_matrix.append(label_prob)

    coverage_matrix = np.array(coverage_matrix).T  # shape (n_samples, 17)
    return coverage_matrix

def format_llm_response_with_probs(input_text, coverage_probs):
    # Convert to percentage
    coverage_pct = coverage_probs * 100
    lines = []
    for i in range(17):
        lines.append(f"SDG {i+1}: {coverage_pct[i]:.1f}%")
    coverage_text = "\n".join(lines)

    return f"""### Instruction:
Predict the coverage (in %) for each SDG for the given text.

### Input:
{input_text}

### Response:
{coverage_text}
"""

# Example usage
example_texts = [
    "This project tackles climate action, so we mention renewable energy and reduced emissions.",
    "A local initiative focuses on zero hunger and quality education for children."
]

print("\n===== Example Coverage Probabilities =====")
cov_probs = predict_coverage_prob(best_model_name, example_texts)

for txt, row in zip(example_texts, cov_probs):
    print(format_llm_response_with_probs(txt, row))
    print("=" * 70)
